# Logistic Regression — Definition + Full Derivations (Sigmoid, Likelihood, MLE, Log-loss)

> **Goal (binary classification):** model a probability $P(y=1\mid x)$ for $y\in\{0,1\}$ given features $x\in\mathbb{R}^d$.

---

## 1) What Logistic Regression *is* (definition)
Logistic regression is a **probabilistic linear classifier**. It assumes the **log-odds** (logit) of the positive class is a linear function of the input:

$$
\text{logit}(p(x)) \;=\; \log\frac{p(x)}{1-p(x)} \;=\; z(x) \;=\; w^\top x + b
$$

where:
- $p(x)=P(y=1\mid x)$
- $w\in\mathbb{R}^d$ are weights, $b\in\mathbb{R}$ is bias/intercept
- $z(x)$ is the **linear score** (also called the *logit*)

Then the predicted probability is obtained by applying the **sigmoid** function:
$$
p(x)=\sigma(z)=\frac{1}{1+e^{-z}}
$$

---

## 2) Sigmoid function: definition and derivation (from log-odds)
### 2.1 Definition
The **sigmoid / logistic** function is:
$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$
It maps any real number $z\in\mathbb{R}$ to a probability in $(0,1)$.

### 2.2 Derivation of the sigmoid from the logit assumption
Start from the logistic regression modeling assumption:
$$
\log\frac{p}{1-p}=z
$$
Exponentiate both sides:
$$
\frac{p}{1-p}=e^{z}
$$
Solve for $p$:
$$
p=e^z(1-p) \;\Rightarrow\; p=e^z - e^z p
$$
Bring $p$ terms together:
$$
p + e^z p = e^z \;\Rightarrow\; p(1+e^z)=e^z
$$
Divide both sides by $(1+e^z)$:
$$
p=\frac{e^z}{1+e^z}=\frac{1}{1+e^{-z}}=\sigma(z)
$$
So, **if log-odds are linear, the probability must be sigmoid of a linear score**.

### 2.3 Derivative of the sigmoid (useful for optimization)
Let $\sigma(z)=\frac{1}{1+e^{-z}}$. Differentiate:
$$
\sigma'(z)=\frac{e^{-z}}{(1+e^{-z})^2}
$$
Now rewrite using $\sigma(z)$ itself:
- $\sigma(z)=\frac{1}{1+e^{-z}}$
- $1-\sigma(z)=\frac{e^{-z}}{1+e^{-z}}$
Multiply them:
$$
\sigma(z)(1-\sigma(z))=\frac{1}{1+e^{-z}}\cdot\frac{e^{-z}}{1+e^{-z}}=\frac{e^{-z}}{(1+e^{-z})^2}
$$
Therefore:
$$
\boxed{\sigma'(z)=\sigma(z)\bigl(1-\sigma(z)\bigr)}
$$

---

## 3) Probability model for the labels (Bernoulli)
In binary classification, for each example $i$:
- $y_i\in\{0,1\}$
- $p_i=P(y_i=1\mid x_i)=\sigma(z_i)$ where $z_i=w^\top x_i+b$

The conditional distribution of $y_i$ given $x_i$ is **Bernoulli**:
$$
P(y_i\mid x_i;w,b)=p_i^{y_i}(1-p_i)^{1-y_i}
$$
This compactly represents both cases:
- if $y_i=1$, probability is $p_i$
- if $y_i=0$, probability is $1-p_i$

---

## 4) Likelihood (dataset)
Assume the $m$ training examples are **i.i.d.** (independent and identically distributed).
The **likelihood** of parameters $(w,b)$ given the dataset is the product of individual probabilities:
$$
\mathcal{L}(w,b)=\prod_{i=1}^{m} P(y_i\mid x_i;w,b)=\prod_{i=1}^{m} p_i^{y_i}(1-p_i)^{1-y_i}
$$
where $p_i=\sigma(w^\top x_i+b)$.

### Interpretation
- A *larger* likelihood means the parameters assign higher probability to the observed labels.
- Products can underflow numerically, so we typically use the log-likelihood.

---

## 5) Log-likelihood (derivation)
Take the log of the likelihood:
$$
\ell(w,b)=\log\mathcal{L}(w,b)=\sum_{i=1}^{m}\log\Big(p_i^{y_i}(1-p_i)^{1-y_i}\Big)
$$
Use log rules ($\log(ab)=\log a+\log b$ and $\log(a^c)=c\log a$):
$$
\ell(w,b)=\sum_{i=1}^{m}\Big[y_i\log p_i+(1-y_i)\log(1-p_i)\Big]
$$
This is the standard **log-likelihood** for Bernoulli logistic regression.

---

## 6) Maximum Likelihood Estimation (MLE)
**MLE** chooses parameters that maximize the likelihood (equivalently maximize log-likelihood):
$$
(w^*,b^*)=\arg\max_{w,b}\;\mathcal{L}(w,b)=\arg\max_{w,b}\;\ell(w,b)
$$
Maximizing $\ell$ is preferred because it turns products into sums and is numerically stable.

---
### 6.1 From “maximize log-likelihood” to “minimize a loss”
Define the **negative log-likelihood (NLL)**:
$$
\text{NLL}(w,b)=-\ell(w,b)=-\sum_{i=1}^{m}\Big[y_i\log p_i+(1-y_i)\log(1-p_i)\Big]
$$
Then MLE is equivalent to:
$$
(w^*,b^*)=\arg\min_{w,b}\;\text{NLL}(w,b)
$$
Often we use the **average** NLL (divide by $m$) so the scale doesn’t grow with dataset size.

---
### 6.2 Parameter optimization using Gradient Descent (derivation + learning rule)
To actually find $(w,b)$ that minimize the average loss, we typically use **gradient descent** (or its variants).

#### Step 1: Define the per-example loss (binary cross-entropy)
For one example $(x_i,y_i)$ with $z_i=w^\top x_i+b$ and $p_i=\sigma(z_i)$:
$$
\mathcal{J}_i(w,b)=-\Big[y_i\log(p_i)+(1-y_i)\log(1-p_i)\Big]
$$
The average dataset loss is:
$$
\mathcal{J}(w,b)=\frac{1}{m}\sum_{i=1}^{m}\mathcal{J}_i(w,b)
$$

#### Step 2: Key derivative $\;\frac{\partial \mathcal{J}_i}{\partial z_i}=p_i-y_i$
We derive it using the chain rule:
- $p_i=\sigma(z_i)$ and $\frac{dp_i}{dz_i}=p_i(1-p_i)$ (from Section 2.3)
- $\mathcal{J}_i=-\big[y_i\log p_i+(1-y_i)\log(1-p_i)\big]$

First differentiate w.r.t. $p_i$:
$$
\frac{\partial \mathcal{J}_i}{\partial p_i}=-\Big(\frac{y_i}{p_i}-\frac{1-y_i}{1-p_i}\Big)
$$
Now apply chain rule:
$$
\frac{\partial \mathcal{J}_i}{\partial z_i}=\frac{\partial \mathcal{J}_i}{\partial p_i}\cdot\frac{\partial p_i}{\partial z_i}
$$
Substitute $\frac{\partial p_i}{\partial z_i}=p_i(1-p_i)$ and simplify:
$$
\frac{\partial \mathcal{J}_i}{\partial z_i}=-\Big(\frac{y_i}{p_i}-\frac{1-y_i}{1-p_i}\Big)\,p_i(1-p_i)=-(y_i(1-p_i)-(1-y_i)p_i)=p_i-y_i
$$
So the crucial result is:
$$
\boxed{\;\frac{\partial \mathcal{J}_i}{\partial z_i}=p_i-y_i\;}
$$

#### Step 3: Gradients w.r.t. parameters $w$ and $b$
Because $z_i=w^\top x_i+b$:
- $\frac{\partial z_i}{\partial w}=x_i$
- $\frac{\partial z_i}{\partial b}=1$

Thus for one example:
$$
\frac{\partial \mathcal{J}_i}{\partial w}=\frac{\partial \mathcal{J}_i}{\partial z_i}\cdot\frac{\partial z_i}{\partial w}=(p_i-y_i)x_i
$$
$$
\frac{\partial \mathcal{J}_i}{\partial b}=\frac{\partial \mathcal{J}_i}{\partial z_i}\cdot\frac{\partial z_i}{\partial b}=(p_i-y_i)
$$
Average over all $m$ samples (batch gradient):
$$
\boxed{\;\nabla_w\mathcal{J}(w,b)=\frac{1}{m}\sum_{i=1}^{m}(p_i-y_i)x_i\;}
$$
$$
\boxed{\;\frac{\partial \mathcal{J}(w,b)}{\partial b}=\frac{1}{m}\sum_{i=1}^{m}(p_i-y_i)\;}
$$

#### Step 4: Gradient Descent learning rule (update equations)
With learning rate $\alpha>0$, **batch gradient descent** updates are:
$$
\boxed{\;w \leftarrow w-\alpha\,\nabla_w\mathcal{J}(w,b)\;}
$$
$$
\boxed{\;b \leftarrow b-\alpha\,\frac{\partial \mathcal{J}(w,b)}{\partial b}\;}
$$
i.e.
$$
w \leftarrow w-\alpha\Big(\frac{1}{m}\sum_{i=1}^{m}(p_i-y_i)x_i\Big),\qquad
b \leftarrow b-\alpha\Big(\frac{1}{m}\sum_{i=1}^{m}(p_i-y_i)\Big)
$$

#### Mini-batch / Stochastic variants (same rule, fewer samples)
- **SGD:** use one sample at a time: $w\leftarrow w-\alpha(p_i-y_i)x_i$, $b\leftarrow b-\alpha(p_i-y_i)$.
- **Mini-batch GD:** use a small batch $B$ of size $|B|$: replace $m$ by $|B|$ and sum only over $i\in B$.

---

## 7) Binary Cross-Entropy Loss / Log-loss / Logistic Loss (derivation)
### 7.1 Definition
The **binary cross-entropy** (also called **log-loss** or **logistic loss**) for one example is:
$$
\mathcal{J}_i = -\Big[y_i\log(p_i)+(1-y_i)\log(1-p_i)\Big]
$$
For a dataset (average loss):
$$
\boxed{\;\mathcal{J}(w,b)=\frac{1}{m}\sum_{i=1}^{m} \mathcal{J}_i= -\frac{1}{m}\sum_{i=1}^{m}\Big[y_i\log p_i+(1-y_i)\log(1-p_i)\Big]\;}
$$

### 7.2 Why this is the right loss for logistic regression (derivation from MLE)
From Section 5, the Bernoulli log-likelihood is:
$$
\ell(w,b)=\sum_{i=1}^{m}\Big[y_i\log p_i+(1-y_i)\log(1-p_i)\Big]
$$
MLE maximizes $\ell(w,b)$. This is equivalent to minimizing $-\ell(w,b)$, i.e. the negative log-likelihood:
$$
-\ell(w,b)=\sum_{i=1}^{m} -\Big[y_i\log p_i+(1-y_i)\log(1-p_i)\Big]
$$
Divide by $m$ (optional scaling) and you get exactly the **binary cross-entropy / log-loss**.

### 7.3 Intuition of the log-loss terms
- If $y=1$, loss is $-\log(p)$: confident correct predictions ($p\to 1$) give near-zero loss; confident wrong ($p\to 0$) gives huge loss.
- If $y=0$, loss is $-\log(1-p)$: confident correct ($p\to 0$) gives near-zero loss; confident wrong ($p\to 1$) gives huge loss.

### 7.4 Why do we *minimize* log-loss?
There are multiple equivalent (and important) reasons:

1) **MLE turns into minimization by a minus sign**
- The “natural” objective from probability theory is to **maximize** the log-likelihood $\ell(w,b)$.
- In optimization (and most ML libraries), we prefer minimizing, so we define the loss as **negative log-likelihood**:
$$
\mathcal{J}(w,b)= -\frac{1}{m}\,\ell(w,b)
$$
Minimizing $\mathcal{J}$ is exactly the same as maximizing $\ell$ (they have the same optimizer).

2) **Minimizing log-loss makes the observed labels as probable as possible**
Because $\mathcal{J}$ comes from the Bernoulli model, lowering $\mathcal{J}$ means increasing
$$
\prod_{i=1}^m P(y_i\mid x_i;w,b)
$$
So the trained model is the one that assigns the **highest probability** to the labels we actually saw.

3) **Cross-entropy / KL-divergence view (information theory)**
For one example, the true label distribution is Bernoulli with parameter $y\in\{0,1\}$ (a “one-hot” Bernoulli). The cross-entropy between the true distribution $q$ and predicted distribution $p$ is:
$$
H(q,p)= -\big[y\log(p)+(1-y)\log(1-p)\big]
$$
So log-loss is literally **cross-entropy**. Also,
$$
H(q,p)=H(q)+D_{\mathrm{KL}}(q\,\|\,p)
$$
Since $H(q)$ does not depend on the model, minimizing log-loss is equivalent to minimizing
$$
D_{\mathrm{KL}}(q\,\|\,p)
$$
i.e. making the model’s predicted distribution $p$ as close as possible to the true distribution $q$.

4) **It strongly penalizes confident wrong predictions**
If the model says $p\approx 0$ when $y=1$, then $-\log(p)$ is huge. This discourages overconfident incorrect probabilities and improves **probability calibration**.

---

## 8) Likelihood vs probability (quick distinction)
- **Probability**: treat parameters as fixed and ask how likely is the data/label outcome. Example: $P(y=1\mid x;w,b)=\sigma(w^\top x+b)$.
- **Likelihood**: treat observed data as fixed and view the same expression as a function of parameters. Example: $\mathcal{L}(w,b)=\prod_i p_i^{y_i}(1-p_i)^{1-y_i}$.
They use the same formula but answer different questions.

---

## 9) Summary table
| Concept | Definition (in words) | Key formula | Notes |
|---|---|---|---|
| Logistic Regression | Models $P(y=1\mid x)$ via a linear log-odds | $\log\frac{p}{1-p}=w^\top x+b$ | Linear decision boundary at $p=0.5$ ($z=0$) |
| Sigmoid / Logistic Function | Maps real score to $(0,1)$ probability | $\sigma(z)=\frac{1}{1+e^{-z}}$ | Derived by solving the logit equation |
| Sigmoid derivative | Slope of sigmoid | $\sigma'(z)=\sigma(z)(1-\sigma(z))$ | Used in gradient calculations |
| Probability (Bernoulli) | Probability of label given $x$ | $P(y\mid x)=p^y(1-p)^{1-y}$ | Here $p=\sigma(w^\top x+b)$ |
| Likelihood | Joint probability of observed labels as a function of parameters | $\mathcal{L}=\prod_i p_i^{y_i}(1-p_i)^{1-y_i}$ | Product over data points (i.i.d.) |
| Log-likelihood | Log of likelihood (turn product to sum) | $\ell=\sum_i [y_i\log p_i+(1-y_i)\log(1-p_i)]$ | Easier + numerically stable |
| MLE | Chooses parameters maximizing likelihood | $(w^*,b^*)=\arg\max \ell(w,b)$ | Equivalent to minimizing NLL |
| NLL / Log-loss / BCE | Loss minimized in training logistic regression | $\mathcal{J}=-\frac{1}{m}\sum_i [y_i\log p_i+(1-y_i)\log(1-p_i)]$ | Exactly $-\frac{1}{m}\ell(w,b)$ |
| Gradient Descent updates | Optimizes $(w,b)$ by iterative descent | $w\leftarrow w-\alpha\frac{1}{m}\sum_i (p_i-y_i)x_i$; $b\leftarrow b-\alpha\frac{1}{m}\sum_i(p_i-y_i)$ | SGD / mini-batch use subset of samples |